In [ ]:
%pip install -qU langchain langchain-community langchain-ollama pymupdf faiss-cpu

In [ ]:
# 1. Install the Ollama background engine server binary
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Spawn the server process in the background
import subprocess
import time
print("Launching Ollama background service...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Give the binary engine 5 seconds to bind to port 11434

# 3. Pull the required models into the server cache
print("Downloading embedding model...")
!ollama pull nomic-embed-text
print("Downloading LLaMA 3 model...")
!ollama pull llama3
print("\nSystem ready! You can now run your main RAG index loop.")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Launching Ollama background service...



System ready! You can now run your main RAG index loop.


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

def build_and_save_vector_store(file_paths, index_dir="faiss_index"):
    documents = []
    for path in file_paths:
        if os.path.exists(path):
            loader = PyMuPDFLoader(path)
            documents.extend(loader.load())
        else:
            print(f"Warning: File path not found: {path}")

    print(f"Total pages loaded: {len(documents)}")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Total structured chunks created: {len(chunks)}")

    embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
    vector_store = FAISS.from_documents(chunks, embeddings_model)
    vector_store.save_local(index_dir)
    print(f"FAISS index successfully built with {vector_store.index.ntotal} vectors.")
    return vector_store

def initialize_reranker_model():
    model_name = "BAAI/bge-reranker-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    return tokenizer, model

def execute_cross_encoder_rerank(query, documents, tokenizer, model, top_n=5):
    if not documents:
        return []

    pairs = [[query, doc.page_content] for doc in documents]
    with torch.no_grad():
        inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors="pt", max_length=512)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        scores = model(**inputs).logits.view(-1).float().tolist()

    scored_docs = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    reranked_docs = [doc for doc, score in scored_docs[:top_n]]
    return reranked_docs

def initialize_retriever(index_dir="faiss_index", k_override=5):
    embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
    loaded_vector_store = FAISS.load_local(
        folder_path=index_dir,
        embeddings=embeddings_model,
        allow_dangerous_deserialization=True
    )
    retriever = loaded_vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": k_override * 3, "fetch_k": k_override * 5}
    )
    return retriever

In [ ]:
import json
from langchain_ollama import ChatOllama

def extract_atomic_claims(draft_answer):
    llm = ChatOllama(model="llama3", temperature=0.0, format="json")
    system_instruction = (
        "You are a linguistic parsing engine. Break down the provided text into a list of individual, "
        "atomic factual assertions. Each claim must be standalone, independent, and contain exactly one fact. "
        "Do not include meta-commentary. Output a strict JSON object matching this schema precisely:\n"
        "{\n"
        "  'atomic_claims': list of strings\n"
        "}"
    )
    user_prompt = f"Text to analyze:\n{draft_answer}"
    try:
        response = llm.invoke([
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt}
        ])
        return json.loads(response.content).get("atomic_claims", [])
    except Exception:
        return []

def verify_single_claim(claim, retrieved_documents):
    llm = ChatOllama(model="llama3", temperature=0.0, format="json")
    context_text = "\n".join([doc.page_content for doc in retrieved_documents])
    system_instruction = (
        "You are a strict binary factual auditor. Verify if the given target claim is explicitly supported "
        "by the provided reference context text. You must be deeply conservative: if the context does not "
        "mention the claim or contradicts it, the verification fails.\n"
        "Output a strict JSON object matching this schema precisely:\n"
        "{\n"
        "  'verdict': boolean (true if fully supported, false if unsupported/contradicted),\n"
        "  'contradiction_detected': boolean,\n"
        "  'citation_source': string (quote the exact sentence from context that supports it, or 'none')\n"
        "}"
    )
    user_prompt = f"Reference Context:\n{context_text}\n\nTarget Claim to Verify: {claim}"
    try:
        response = llm.invoke([
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt}
        ])
        return json.loads(response.content)
    except Exception:
        return {"verdict": False, "contradiction_detected": False, "citation_source": "none"}

In [ ]:
def check_cross_document_contradictions(documents):
    if len(documents) < 2:
        return False, []
    llm = ChatOllama(model="llama3", temperature=0.0, format="json")
    conflicts = []
    system_instruction = (
        "You are a factual contradiction auditor. Compare the two provided context blocks and determine if they "
        "contain direct factual collisions, opposing assertions, or mutually exclusive timelines.\n"
        "Output a strict JSON object matching this schema precisely:\n"
        "{\n"
        "  'contradiction_detected': boolean,\n"
        "  'conflict_explanation': string (explain the collision details, or 'none')\n"
        "}"
    )
    for i in range(len(documents)):
        for j in range(i + 1, len(documents)):
            source_i = documents[i].metadata.get("source", "Doc A").split("/")[-1]
            source_j = documents[j].metadata.get("source", "Doc B").split("/")[-1]
            if source_i == source_j:
                continue
            user_prompt = f"Block A Reference Source: {source_i}\nContent:\n{documents[i].page_content}\n\nBlock B Reference Source: {source_j}\nContent:\n{documents[j].page_content}"
            try:
                response = llm.invoke([
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": user_prompt}
                ])
                res = json.loads(response.content)
                if res.get("contradiction_detected", False):
                    conflicts.append({
                        "conflict_pair": f"{source_i} <--> {source_j}",
                        "diagnostic_explanation": res.get("conflict_explanation")
                    })
            except Exception:
                continue
    return len(conflicts) > 0, conflicts

def execute_mathematical_verification(query, retrieved_documents):
    generator_llm = ChatOllama(model="llama3", temperature=0.1)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_documents])
    generation_prompt = (
        f"Answer the user query based strictly on the provided context chunks. If the context does not contain "
        f"the information needed to answer, state clearly that the context is insufficient.\n\n"
        f"Context:\n{context_text}\n\nQuery: {query}\n\nAnswer:"
    )
    draft_answer = generator_llm.invoke(generation_prompt).content.strip()

    cross_conflict_found, active_conflicts = check_cross_document_contradictions(retrieved_documents)
    claims = extract_atomic_claims(draft_answer)
    refusal_keywords = ["insufficient context", "cannot be answered", "does not contain", "not possible to answer", "unable to answer", "no information"]
    is_refusal = any(keyword in draft_answer.lower() for keyword in refusal_keywords) or (not claims)

    if is_refusal:
        return {
            "evidence_coverage_percentage": 0,
            "context_relevance_score": 0.0,
            "hallucination_risk_index": 1.0,
            "detected_unsupported_claims": ["All assertions ungrounded due to systemic context refusal."],
            "evidence_sufficiency": False,
            "failure_localization_mode": "insufficient_context",
            "failure_explanation": "Source context does not contain the required factual data to address the target query.",
            "structured_reasoning_trace": [{
                "claim_id": 1,
                "claim_text": "System generated an explicit context refusal statement.",
                "status": "REFUSAL_HALT",
                "source_citation": "none"
            }],
            "final_answer": f"Operation aborted. Reason: Source context contains insufficient information to answer the query directly."
        }

    supported_count = 0
    claim_contradiction_found = False
    unsupported_claims_list = []
    reasoning_graph_trace = []

    for idx, claim in enumerate(claims):
        audit = verify_single_claim(claim, retrieved_documents)
        is_supported = audit.get("verdict", False)
        is_contradicted = audit.get("contradiction_detected", False)

        if is_supported:
            supported_count += 1
            status = "SUPPORTED"
        else:
            unsupported_claims_list.append(claim)
            status = "CONTRADICTED" if is_contradicted else "UNSUPPORTED"
            if is_contradicted:
                claim_contradiction_found = True

        reasoning_graph_trace.append({
            "claim_id": idx + 1,
            "claim_text": claim,
            "status": status,
            "source_citation": audit.get("citation_source", "none")
        })

    total_claims = len(claims)
    evidence_coverage = int((supported_count / total_claims) * 100)
    hallucination_risk = round(1.0 - (supported_count / total_claims), 2)
    context_relevance = round(supported_count / total_claims, 2)

    system_wide_contradiction = claim_contradiction_found or cross_conflict_found
    evidence_sufficiency = (evidence_coverage >= 80) and not system_wide_contradiction and (len(unsupported_claims_list) == 0)

    failure_mode = "none"
    explanation = "none"
    if not evidence_sufficiency:
        if system_wide_contradiction:
            failure_mode = "contradictory_sources"
            explanation = f"Cross-document logic data collision caught. Active Conflicts: {json.dumps(active_conflicts)}"
        elif evidence_coverage == 0:
            failure_mode = "retrieval_drift"
            explanation = "FAISS indices returned semantically peripheral vectors containing zero overlapping facts."
        elif evidence_coverage < 50:
            failure_mode = "weak_evidence_density"
            explanation = f"Low token informational density detected. Only {evidence_coverage}% of assertions verified."
        else:
            failure_mode = "reasoning_break"
            explanation = "The response contains ungrounded logical leaps or unsupported peripheral claims."

    return {
        "evidence_coverage_percentage": evidence_coverage,
        "context_relevance_score": context_relevance,
        "hallucination_risk_index": hallucination_risk,
        "detected_unsupported_claims": unsupported_claims_list,
        "evidence_sufficiency": evidence_sufficiency,
        "failure_localization_mode": failure_mode,
        "failure_explanation": explanation,
        "structured_reasoning_trace": reasoning_graph_trace,
        "final_answer": draft_answer if evidence_sufficiency else f"Operation aborted. Reason: {explanation}"
    }

In [ ]:
def load_global_failure_memory(file_path="global_failure_memory.json"):
    if os.path.exists(file_path):
        try:
            with open(file_path, "r") as f:
                return json.load(f)
        except Exception:
            return []
    return []

def append_to_global_failure_memory(query, failure_mode, diagnostic, file_path="global_failure_memory.json"):
    memory = load_global_failure_memory(file_path)
    memory.append({
        "historical_query": query,
        "confirmed_failure_mode": failure_mode,
        "system_diagnostic": diagnostic
    })
    with open(file_path, "w") as f:
        json.dump(memory, f, indent=2)

def check_past_vulnerabilities(query, file_path="global_failure_memory.json"):
    memory = load_global_failure_memory(file_path)
    if not memory:
        return False
    llm = ChatOllama(model="llama3", temperature=0.0, format="json")
    system_instruction = (
        "You are an analytical vulnerability auditor. Check if the incoming query targets a known weak evidence region "
        "or shares structural failure patterns with past execution breakdowns listed in the provided history log.\n"
        "Output a strict JSON object matching this schema precisely:\n"
        "{\n"
        "  'is_vulnerable': boolean\n"
        "}"
    )
    user_prompt = f"Current Query: {query}\n\nPast Failure History Log:\n{json.dumps(memory[-10:], indent=2)}"
    try:
        response = llm.invoke([
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt}
        ])
        return json.loads(response.content).get("is_vulnerable", False)
    except Exception:
        return False

def print_structured_evidence_graph(query, manifest):
    print("\n" + "="*70)
    print(" DETERMINISTIC EVIDENCE DEPENDENCY AND LINEAGE LATTICE GRAPH")
    print("="*70)
    print(f" Target Query: '{query}'")
    print(" │")
    trace_list = manifest.get("structured_reasoning_trace", [])
    for idx, node in enumerate(trace_list):
        is_last_claim = (idx == len(trace_list) - 1)
        claim_prefix = " └──" if is_last_claim else " ├──"
        pipe_prefix = "    " if is_last_claim else " │  "
        status = node.get("status", "UNSUPPORTED")
        claim_text = node.get("claim_text", "")
        citation = node.get("source_citation", "none")
        if status == "SUPPORTED":
            color_status = f"[\033[92m{status}\033[0m]"
        elif status == "REFUSAL_HALT":
            color_status = f"[\033[93m{status}\033[0m]"
        else:
            color_status = f"[\033[91m{status}\033[0m]"
        print(f"{claim_prefix}  Assertion {node.get('claim_id')}: \"{claim_text}\" -> {color_status}")
        if citation != "none":
            print(f"{pipe_prefix}  Text Provenance Anchor: \"{citation[:90]}...\"")
        else:
            print(f"{pipe_prefix}  Provenance Isolation: Zero deterministic weight match found in data blocks.")
    print(" │")
    print(f" └──  Mitigation Routing Action Mode: {manifest.get('failure_localization_mode').upper()}")
    print("="*70 + "\n")

def run_adaptive_reasoning_agent(query, tokenizer, model, index_dir="faiss_index"):
    max_retries = 2
    retry_count = 0
    current_query = query
    local_session_failures = []

    print(f"\n[AGENT START] Checking cross-session failure memory ledger for query: '{query}'")
    is_known_vulnerability = check_past_vulnerabilities(query)

    k_val = 10 if is_known_vulnerability else 5
    if is_known_vulnerability:
        print(f"[ADAPTIVE TRIGGER] Query hits historical risk signature. Pre-scaling candidate space to depth k=10.")
    else:
        print(f"[STANDALONE RUN] Query path clean of known historical vulnerabilities. Operating baseline depth k=5.")

    retriever_instance = initialize_retriever(index_dir, k_override=k_val)

    while retry_count <= max_retries:
        print(f"\n--- [Stage 1: Vector Retrieval & Cross-Encoder Rerank Run {retry_count + 1}] ---")
        candidate_chunks = retriever_instance.invoke(current_query)
        retrieved_chunks = execute_cross_encoder_rerank(query, candidate_chunks, tokenizer, model, top_n=5)
        print(f"FAISS candidate vectors scaled. Cross-Encoder distilled down to 5 highest semantic ranking chunks.")

        print("\n--- [Stage 2: Running Deterministic Verification Engine] ---")
        manifest = execute_mathematical_verification(query, retrieved_chunks)
        print(f"Calculated Boundary -> Sufficiency: {manifest['evidence_sufficiency']} | Grounded Coverage: {manifest['evidence_coverage_percentage']}%")

        print_structured_evidence_graph(query, manifest)

        if manifest["evidence_sufficiency"]:
            print("\n[AGENT SUCCESS] System verified mathematical integrity. Outputting final trusted trace.")
            return {
                "status": "VERIFIED_SUCCESS",
                "execution_metadata": {"total_retries_attempted": retry_count, "failure_history": local_session_failures},
                "verification_metrics": {
                    "coverage_score": manifest["evidence_coverage_percentage"],
                    "relevance_score": manifest["context_relevance_score"],
                    "hallucination_risk": manifest["hallucination_risk_index"],
                    "unsupported_claims_count": len(manifest["detected_unsupported_claims"])
                },
                "reasoning_graph": manifest["structured_reasoning_trace"],
                "output": manifest["final_answer"]
            }

        print(f"\n[AGENT WARNING] Anomaly validation limits reached. Failure Mode Triggered: {manifest['failure_localization_mode']}")
        print(f"Diagnostic Report: {manifest['failure_explanation']}")

        failure_summary = {
            "attempt": retry_count + 1,
            "mode": manifest["failure_localization_mode"],
            "diagnostic": manifest["failure_explanation"]
        }
        local_session_failures.append(failure_summary)

        if retry_count == max_retries:
            print("\n[LEDGER WRITE] Retries exhausted. Writing failure signature directly to persistent ledger.")
            append_to_global_failure_memory(query, manifest["failure_localization_mode"], manifest["failure_explanation"])

        if retry_count < max_retries:
            print("\n--- [Stage 3: Reflective Query Refinement Loop Activated] ---")
            refinement_llm = ChatOllama(model="llama3", temperature=0.4)
            refinement_prompt = (
                f"You are an adaptive retrieval strategist. The initial search query '{query}' failed due to lack of specific evidence.\n"
                f"The failure diagnosis indicates: {manifest['failure_explanation']}.\n"
                f"Generate a single line of descriptive, natural language keywords and synonyms that target the missing facts directly. "
                f"Do NOT use boolean operators like AND, OR, quotes, or heavy nested brackets. Output ONLY the raw search string."
            )
            current_query = refinement_llm.invoke(refinement_prompt).content.strip()
            print(f"Reformulated query generated for deep extraction: '{current_query}'")

        retry_count += 1

    print("\n[AGENT CRITICAL TERMINATION] Maximum execution attempts exhausted without passing verification thresholds.")
    return {
        "status": "FAILED_VERIFICATION_UNTRUSTED",
        "execution_metadata": {"total_retries_attempted": max_retries, "failure_history": local_session_failures},
        "verification_metrics": {
            "coverage_score": manifest["evidence_coverage_percentage"],
            "relevance_score": manifest["context_relevance_score"],
            "hallucination_risk": manifest["hallucination_risk_index"],
            "unsupported_claims": manifest["detected_unsupported_claims"]
        },
        "reasoning_graph": manifest["structured_reasoning_trace"],
        "output": f"Operation aborted. Reason: {manifest['failure_explanation']}"
    }

In [ ]:
if __name__ == "__main__":
    pdf_file_paths = [
        "/content/2010.08678v3.pdf",
        "/content/2302.04761v1.pdf",
        "/content/2305.10601v2.pdf"
    ]

    print("Loading local neural Cross-Encoder model weights...")
    tok, mod = initialize_reranker_model()

    test_query = "What are the success rates of IO, CoT, and CoT-SC prompting methods as referenced in Table 2?"

    final_trusted_result = run_adaptive_reasoning_agent(test_query, tok, mod)

    print("\n================== FINAL SYSTEM SUMMARY REPORT ==================")
    print(json.dumps(final_trusted_result, indent=2))

Loading local neural Cross-Encoder model weights...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[AGENT START] Checking cross-session failure memory ledger for query: 'What are the success rates of IO, CoT, and CoT-SC prompting methods as referenced in Table 2?'
[STANDALONE RUN] Query path clean of known historical vulnerabilities. Operating baseline depth k=5.

--- [Stage 1: Vector Retrieval & Cross-Encoder Rerank Run 1] ---
FAISS candidate vectors scaled. Cross-Encoder distilled down to 5 highest semantic ranking chunks.

--- [Stage 2: Running Deterministic Verification Engine] ---
Calculated Boundary -> Sufficiency: True | Grounded Coverage: 100%

 DETERMINISTIC EVIDENCE DEPENDENCY AND LINEAGE LATTICE GRAPH
 Target Query: 'What are the success rates of IO, CoT, and CoT-SC prompting methods as referenced in Table 2?'
 │
 ├──  Assertion 1: "The success rate of IO prompting method is 7.3%." -> [SUPPORTED]
 │    Text Provenance Anchor: "As shown in Table 2, IO, CoT, and CoT-SC prompting methods perform badly on the task, achi..."
 ├──  Assertion 2: "The success rate of CoT prompti